In [ ]:
import h5py
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

from gwpy.timeseries import TimeSeries
from gwpy.frequencyseries import FrequencySeries
from ripple.waveforms.IMRPhenomD import gen_IMRPhenomD_hphc

## Load generated data

True, time-delayed O3a detector noise plus an injected synthetic signal.
Truth parameters are kept under `/truth` for cross-checks.

In [ ]:
arrays = {}
with h5py.File("test_bbh_v3.h5", "r") as f:
    f.visititems(
        lambda name, obj: arrays.update({name: obj[...]})
        if isinstance(obj, h5py.Dataset) else None
    )

for name, arr in arrays.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

ifo = "H1"
color = {"H1": "#ee0000", "L1": "#4ba6ff"}[ifo]
fs = 4096
N = arrays[f"{ifo}/strain"].shape[0]


## Inputs in time and frequency space

In [ ]:
strain_ts = TimeSeries(arrays[f"{ifo}/strain"], t0=0, sample_rate=fs)
psd_fs = FrequencySeries(arrays[f"{ifo}/psd"], f0=0, df=fs / N)

strain_ts.plot(
    title=f"{ifo} strain (signal + O3a noise)", ylabel="Strain", color=color,
).show()

In [ ]:
white = strain_ts.whiten(asd=psd_fs ** 0.5).bandpass(30, 400)
plot = white.plot(color=color)
plot.gca().set_xlim(6.0, 8.0)
plot.show()

qtrans = white.q_transform(frange=(20, 500), outseg=(6.25, 8), whiten=False)
plot = qtrans.plot(figsize=(8, 5), vmin=0, vmax=25)
ax = plot.gca()
ax.set_yscale("log")
ax.set_ylabel("Frequency [Hz]")
plot.colorbar(label="Normalized energy")
plot.show()

## Matched filtering

Build `d_full` and `h_full` on the segment rFFT grid, then run the proper
matched filter against the stored PSD as a baseline.

In [ ]:
df = fs / N
freqs = jnp.fft.rfftfreq(N, d=1.0 / fs)
band = (freqs >= 20) & (freqs <= fs / 2)
f_ref = 20.0

strain = jnp.asarray(np.asarray(strain_ts))
psd_arr = jnp.asarray(np.asarray(psd_fs))

# Truth parameters in ripple's BBH order: [Mc, eta, chi1, chi2, D, tc, phic, iota].
q = float(arrays["truth/mass_ratio"])
theta_truth = jnp.array([
    float(arrays["truth/chirp_mass"]),
    q / (1 + q) ** 2,
    float(arrays["truth/chi1"]),
    float(arrays["truth/chi2"]),
    float(arrays["truth/distance"]),
    float(arrays["truth/tc"]),
    float(arrays["truth/phic"]),
    float(arrays["truth/inclination"]),
])

# Template on the rFFT grid, projected through this IFO's antenna factors.
fplus = float(arrays[f"antenna/{ifo}/fplus"])
fcross = float(arrays[f"antenna/{ifo}/fcross"])
hp, hc = gen_IMRPhenomD_hphc(freqs[band].astype(jnp.float64), theta_truth, f_ref)
h_band = (fplus * hp + fcross * hc).astype(jnp.complex128)
h_full = jnp.zeros_like(freqs, dtype=jnp.complex128).at[band].set(h_band)

# Tukey(alpha=0.01) on the strain matches the PSD's segment-edge treatment
# without eating the merger (only the first/last 1% of samples are tapered).
n_tukey = int(round(0.01 * N / 2))
ramp = 0.5 * (1 - jnp.cos(jnp.pi * (jnp.arange(n_tukey) + 0.5) / n_tukey))
window_td = jnp.concatenate([ramp, jnp.ones(N - 2 * n_tukey), ramp[::-1]])
d_full = jnp.fft.rfft(strain * window_td) / fs

psd_safe = jnp.where(band, psd_arr, jnp.inf)

In [ ]:
inner_hh = 4.0 * df * jnp.real(jnp.sum(jnp.abs(h_full) ** 2 / psd_safe))
truth_snr = float(arrays[f"truth/snr_{ifo}"])
print(f"sqrt<h|h> (stored PSD): {float(jnp.sqrt(inner_hh)):.3f}")
print(f"truth {ifo} SNR        : {truth_snr:.3f}")

## Matched filter


In [ ]:
# z(t) = 4 ∫ conj(d̃) h̃ e^{2πift} / S_n(f) df          # cross-correlation vs shift
# ρ(t) = |z(t)| / sqrt(<h|h>)                            # phase-maximised SNR
#
# Zero-pad the one-sided integrand back to a full length-N complex spectrum
# (negative freqs = 0) before the inverse FFT, so the 4 · fs factor matches
# the continuous-time matched-filter normalisation.
integrand = jnp.conj(d_full) * h_full / psd_safe
z_t = 4.0 * fs * jnp.fft.ifft(
    jnp.zeros(N, dtype=jnp.complex128).at[: integrand.shape[0]].set(integrand)
)
rho_t = jnp.abs(z_t) / jnp.sqrt(inner_hh)
peak_idx = int(jnp.argmax(rho_t))

# The ifft index encodes the *relative* shift τ_peak between data and template,
# with periodic wrap-around: indices > N/2 map to negative shifts. The
# recovered coalescence time in the data is then
#     tc_data = tc_template − τ_peak
tau_peak = (peak_idx if peak_idx < N // 2 else peak_idx - N) / fs
tc_recovered = float(theta_truth[5]) - tau_peak

print(f"<h|h>                : {float(inner_hh):.3f}")
print(f"Matched-filter peak  : {float(rho_t[peak_idx]):.3f}")
print(f"  shift τ_peak       : {tau_peak * 1e3:+.3f} ms")
print(f"  recovered tc       : {tc_recovered:.4f} s   (truth tc: {float(theta_truth[5]):.4f} s)")
print(f"True {ifo} SNR          : {truth_snr:.3f}")


## Fit coalescence time with JAX

Phase-shift the existing `h_full` by `exp(-2πi·f·Δtc)` (FT shift theorem) and
gradient-descend on `-|<d|h_Δtc>|²/<h|h>`. The phase-maximised SNR² has a
main-lobe width of ~`1/f_max ≈ 0.5 ms` around the truth, so we start a few
tenths of a millisecond off-truth — outside that, the gradient flattens out
and you'd want to fall back to the ifft-based time-maximisation.

In [ ]:
@jax.jit
@jax.value_and_grad
def neg_rho_sq_tc(delta_tc):
    h_shifted = h_full * jnp.exp(-2j * jnp.pi * freqs * delta_tc)
    dh = 4.0 * df * jnp.sum(jnp.conj(d_full) * h_shifted / psd_safe)
    return -(jnp.abs(dh) ** 2 / inner_hh)


dtc = jnp.float64(5e-4)              # start 0.5 ms off truth
opt = optax.adam(1e-5)
state = opt.init(dtc)

hist_dtc, hist_rho_tc = [], []
for _ in range(500):
    loss, g = neg_rho_sq_tc(dtc)
    hist_dtc.append(float(dtc))
    hist_rho_tc.append(float(jnp.sqrt(-loss)))
    updates, state = opt.update(g, state)
    dtc = optax.apply_updates(dtc, updates)

tc_template = float(theta_truth[5])
print(f"Initial Δtc : {hist_dtc[0] * 1e3:+.3f} ms   →  ρ = {hist_rho_tc[0]:.3f}")
print(f"Final   Δtc : {hist_dtc[-1] * 1e3:+.3f} ms   →  ρ = {hist_rho_tc[-1]:.3f}")
print(f"Recovered tc: {tc_template + float(dtc):.4f} s   (truth: {tc_template:.4f} s)")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
ax0.plot(np.array(hist_dtc) * 1e3, color="C0")
ax0.axhline(0, ls="--", color="crimson", label="truth Δtc = 0")
ax0.set_xlabel("iteration"); ax0.set_ylabel("Δtc [ms]"); ax0.legend()
ax1.plot(hist_rho_tc, color="C0")
ax1.axhline(truth_snr, ls="--", color="navy", label=f"truth ρ = {truth_snr:.2f}")
ax1.set_xlabel("iteration"); ax1.set_ylabel("ρ"); ax1.legend()
fig.tight_layout(); plt.show()

## Fit chirp mass with JAX

Regenerate the template at each Mc and gradient-descend on
`-max_t |z(t)|² / <h|h>`, i.e. the time-maximised matched-filter SNR² — the
same statistic `fit_waveform.py` uses. Time-maximising via ifft removes the
need to also fit tc, because as Mc moves the merger drifts inside the
segment and `max_t` tracks it automatically.

In [ ]:
def template_for(theta):
    hp, hc = gen_IMRPhenomD_hphc(freqs[band].astype(jnp.float64), theta, f_ref)
    h_b = (fplus * hp + fcross * hc).astype(jnp.complex128)
    return jnp.zeros_like(freqs, dtype=jnp.complex128).at[band].set(h_b)


@jax.jit
@jax.value_and_grad
def neg_rho_sq_mc(mc):
    theta = theta_truth.at[0].set(mc)
    h = template_for(theta)
    hh = 4.0 * df * jnp.real(jnp.sum(jnp.abs(h) ** 2 / psd_safe))
    integrand = jnp.conj(d_full) * h / psd_safe
    pad = jnp.zeros(N, dtype=jnp.complex128).at[: integrand.shape[0]].set(integrand)
    z = 4.0 * fs * jnp.fft.ifft(pad)
    return -(jnp.max(jnp.abs(z) ** 2) / hh)


mc_truth = float(theta_truth[0])
mc = jnp.float64(mc_truth + 1.0)     # start 1 Msun above truth
schedule = optax.cosine_decay_schedule(init_value=0.1, decay_steps=300, alpha=1e-3 / 0.1)
opt = optax.adam(schedule)
state = opt.init(mc)

hist_mc, hist_rho_mc = [], []
for _ in range(300):
    loss, g = neg_rho_sq_mc(mc)
    hist_mc.append(float(mc))
    hist_rho_mc.append(float(jnp.sqrt(-loss)))
    if not np.isfinite(g):
        print(f"non-finite gradient at Mc = {float(mc):.4f}; stopping")
        break
    updates, state = opt.update(g, state)
    mc = optax.apply_updates(mc, updates)

print(f"Initial Mc : {hist_mc[0]:.4f} Msun   →  ρ = {hist_rho_mc[0]:.3f}")
print(f"Final   Mc : {hist_mc[-1]:.4f} Msun   →  ρ = {hist_rho_mc[-1]:.3f}")
print(f"Truth   Mc : {mc_truth:.4f} Msun         (truth ρ = {truth_snr:.3f})")

iters = np.arange(len(hist_mc))
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5),
                               gridspec_kw={"width_ratios": [1, 1.4]})
ax0.plot(iters, hist_mc, color="C0")
ax0.axhline(mc_truth, ls="--", color="crimson", label=f"truth Mc = {mc_truth:.3f}")
ax0.set_xlabel("iteration"); ax0.set_ylabel(r"$\mathcal{M}$ [$M_\odot$]"); ax0.legend()

sc = ax1.scatter(hist_mc, hist_rho_mc, c=iters, cmap="viridis", s=20)
ax1.plot(hist_mc, hist_rho_mc, color="gray", alpha=0.3, lw=1)
ax1.axvline(mc_truth, ls="--", color="crimson", alpha=0.7, label=f"truth Mc = {mc_truth:.3f}")
ax1.axhline(truth_snr, ls="--", color="navy", alpha=0.7, label=f"truth ρ = {truth_snr:.2f}")
ax1.set_xlabel(r"$\mathcal{M}$ [$M_\odot$]"); ax1.set_ylabel("ρ"); ax1.legend()
fig.colorbar(sc, ax=ax1, label="iteration")
fig.tight_layout(); plt.show()